# MoshiRAG retrieval-latency sweep on a Colab T4 (low-memory version)

Runtime → Change runtime type → **T4 GPU**. Run the cells in order. Every step is resumable:
if Colab disconnects, rerun the setup cells (0–5) and then the step you were on — completed runs are skipped.

What runs: MoshiRAG (int8-quantized LM, float16), the same frame-based retrieval-latency injection as the
A100 version, a fixed reference whose embedding is precomputed once. No STT, no retrieval LLM.
See `experiments/async_retrieval_latency/t4/README.md`.

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Settings
Results and the embedding cache go to Google Drive so they survive disconnects.
The Hugging Face cache (~18 GB, or ~30 GB with the encoder) stays on the local disk.

In [ ]:
REPO_URL = "https://github.com/zxx3312/moshi-rag.git"   # your fork
BRANCH = "exp/async-retrieval-latency"
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/moshirag_t4"
else:
    WORK = "/content/moshirag_t4"
os.makedirs(WORK, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_home"   # set before anything imports huggingface_hub
REPO = "/content/moshi-rag"
EXP = f"{REPO}/experiments/async_retrieval_latency"
CACHE = f"{WORK}/reference_cache"
RESULTS = f"{WORK}/results"
print(WORK, CACHE, RESULTS, sep="\n")

## 2. Clone the repository (private fork: use a GitHub token in the URL or `gh auth login`)

In [ ]:
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} {REPO_URL} {REPO}
else:
    # Drop edits made inside Colab (e.g. the hand-applied patch) so the pull cannot conflict.
    # Ignored files (configs/audio link) and everything on Drive are not touched.
    !git -C {REPO} checkout -- . && git -C {REPO} pull --ff-only
!git -C {REPO} log --oneline -1

## 3. Install pinned dependencies (2–4 min; nothing here imports torch in this notebook, so no restart is needed)

In [ ]:
!pip install -q -r {EXP}/t4/requirements-colab.txt

In [ ]:
!pip uninstall -y -q torchvision
!python -c "import torch, bitsandbytes, xformers, transformers, sphn, huggingface_hub; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0)); print('bnb', bitsandbytes.__version__, '| xformers', xformers.__version__, '| transformers', transformers.__version__, '| hub', huggingface_hub.__version__)"
!python -c "import torch, bitsandbytes as bnb; from bitsandbytes import functional as F; w=torch.randn(256,128,device='cuda',dtype=torch.float16); CB,SCB,_=F.int8_vectorwise_quant(w); s=bnb.MatmulLtState(); s.CB,s.SCB,s.has_fp16_weights=CB,SCB,False; x=torch.randn(4,128,device='cuda',dtype=torch.float16); y=bnb.matmul(x,CB,state=s); r=x.float()@w.float().T; print('int8 matmul on T4 OK, rel err', float((y.float()-r).norm()/r.norm()))"
!python -c "from transformers import AutoTokenizer; print('transformers import OK')"


## 4. Hugging Face login
Needed only for step 6 (the reference encoder's tokenizer comes from the gated
`meta-llama/Llama-3.2-3B-Instruct`; only its ~9 MB tokenizer files are downloaded, never the Llama weights).
Use a **read** token whose account has accepted the Llama 3.2 license.

In [ ]:
from getpass import getpass
from huggingface_hub import login, whoami
login(token=getpass("Hugging Face read token: "), add_to_git_credential=False)
print("logged in as", whoami()["name"])


## 5. Make the question WAVs (Piper TTS in a separate venv, CPU, ~1 s per question)

In [ ]:
import json, pathlib, shutil, subprocess, sys
AUDIO = pathlib.Path(f"{WORK}/audio")   # 存在 Drive 上：只生成一次，--resume 时复用
AUDIO.mkdir(parents=True, exist_ok=True)
link = pathlib.Path(f"{EXP}/configs/audio")
if link.is_dir() and not link.is_symlink():      # 旧版单元格生成的 WAV：保留下来
    for f in link.glob("*.wav"):
        if not (AUDIO / f.name).exists():
            shutil.move(str(f), AUDIO / f.name)
    shutil.rmtree(link)
if not link.is_symlink():
    link.symlink_to(AUDIO, target_is_directory=True)

# Piper 装进单独的文件夹（不用 venv：Colab 的 python 常常缺少 ensurepip）
TTS = "/content/tts_pkgs"
VOICE = "/content/piper_voices/en_US-lessac-medium.onnx"
if not os.path.isdir(f"{TTS}/piper"):
    !pip install -q --target {TTS} piper-tts
env = dict(os.environ, PYTHONPATH=TTS)

def run(cmd):
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout, r.stderr)
        raise RuntimeError(f"failed: {' '.join(cmd)}")
    return r.stdout

if not os.path.exists(VOICE):
    run([sys.executable, "-m", "piper.download_voices", "en_US-lessac-medium", "--data-dir", "/content/piper_voices"])
for manifest in [f"{EXP}/configs/samples.jsonl", f"{EXP}/t4/configs/samples_small.jsonl"]:
    for line in open(manifest):
        s = json.loads(line)
        wav = (pathlib.Path(manifest).parent / s["wav"]).resolve()
        if not wav.exists():
            print(run([sys.executable, f"{EXP}/scripts/make_tts_question.py", "--voice", VOICE,
                       "--text", s["question"], "--out", str(wav)]).strip())
!ls -la {AUDIO}


In [ ]:
!python -c "import numpy, torch; print('numpy', numpy.__version__, numpy.__file__); print('torch', torch.__version__)"


## 6. Precompute reference embeddings (one time; separate process, frees the GPU when it exits)
Downloads the MoshiRAG checkpoint (15.4 GB) and the ARC encoder (12.1 GB). On a T4 the encoder autocasts to
float16 (no bf16 support); the official encoder uses bfloat16. For exact parity, compute the cache on an
A100 instead and copy it to `CACHE`.

In [ ]:
p = f"{EXP}/t4/precompute_reference_embeddings.py"
s = open(p).read()
anchor = '    logger.info("ARC weights assigned (%d unexpected keys ignored)", len(result.unexpected_keys))\n'
if "cond.requires_grad_(False)" not in s:
    assert anchor in s, "anchor not found"
    s = s.replace(anchor, anchor + "    cond.requires_grad_(False)\n")
    open(p, "w").write(s)
print("patched" if "cond.requires_grad_(False)" in open(p).read() else "NOT patched")


In [ ]:
!python {EXP}/t4/precompute_reference_embeddings.py --manifest {EXP}/configs/samples.jsonl {EXP}/t4/configs/samples_small.jsonl --cache-dir {CACHE}

## 7. Step 1 — 1 sample × 1 delay (3 s)
Checks: model loads in 16 GB, `[RET]` → injection works, audio + JSONL are written. Read the memory lines in the output.

In [ ]:
ARGS = f"--reference-cache {CACHE} --output-root {RESULTS} --resume"
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --retrieval-delay 3 --run-name step1_1x1 {ARGS} --dry-run
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --retrieval-delay 3 --run-name step1_1x1 {ARGS}

In [ ]:
import json
run = f"{RESULTS}/step1_1x1"
print(open(f"{run}/run_status.json").read())
for r in map(json.loads, open(f"{run}/results.jsonl")):
    print({k: r[k] for k in ["run_id", "status", "stop_reason", "rag_triggered", "retrieval_trigger_time",
                             "reference_injection_time", "trigger_to_injection_s", "gpu_peak_allocated_mb", "run_wall_s"]})
    print("pre-RAG :", r["pre_rag_transcript"]); print("post-RAG:", r["post_rag_transcript"])
from IPython.display import Audio
Audio(f"{run}/audio/" + r["run_id"] + "_stereo.wav")

## 8. Step 2 — 1 sample × all delays (0, 1, 2, 3, 5, 8 s)

In [ ]:
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --run-name step2_1xall {ARGS}

In [ ]:
!python {EXP}/analysis/analyze_latency.py {RESULTS}/step2_1xall

## 9. Step 3 — small multi-sample sweep (5 samples × 6 delays = 30 runs)
Long: rerun this cell after a disconnect, it resumes. The 5 questions are examples; choose your own evaluation set.

In [ ]:
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/t4/configs/samples_small.jsonl --run-name step3_small {ARGS}

In [ ]:
!python {EXP}/analysis/analyze_latency.py {RESULTS}/step3_small

## GPU memory log

In [ ]:
for line in open(f"{RESULTS}/step1_1x1/logs/gpu_memory.jsonl"):
    r = json.loads(line)
    print(f"{r['event']:28s} allocated {r.get('allocated_mb', 0):8.0f} MB  peak {r.get('peak_allocated_mb', 0):8.0f} MB  device {r.get('device_used_mb', 0):8.0f} MB")

In [ ]:
!git -C {REPO} status --short
!git -C {REPO} diff --stat


## 10. Hard-question pilot (12 SimpleQA questions; delays 0 and 8 s first)
Questions the model is unlikely to know, so its behaviour before the reference arrives matters.
Selection: `scripts/build_simpleqa_pilot.py` (rule-based filter + fixed-seed sample, no hand-picking).
References restate question + gold answer only. Run cells 0, 1, 2, 3 (+ check), 4, 5 first.

In [ ]:
# WAVs for the pilot questions (stored on Drive through the configs/audio link made in step 5)
import json, os, pathlib, subprocess, sys
TTS = "/content/tts_pkgs"
VOICE = "/content/piper_voices/en_US-lessac-medium.onnx"
if not os.path.isdir(f"{TTS}/piper"):
    !pip install -q --target {TTS} piper-tts
env = dict(os.environ, PYTHONPATH=TTS)
if not os.path.exists(VOICE):
    subprocess.run([sys.executable, "-m", "piper.download_voices", "en_US-lessac-medium", "--data-dir", "/content/piper_voices"], env=env, check=True)
HARD = f"{EXP}/configs/samples_hard_pilot.jsonl"
assert pathlib.Path(f"{EXP}/configs/audio").is_symlink(), "run step 5 first (it links configs/audio to Drive)"
for line in open(HARD):
    s = json.loads(line)
    wav = (pathlib.Path(HARD).parent / s["wav"]).resolve()
    if not wav.exists():
        r = subprocess.run([sys.executable, f"{EXP}/scripts/make_tts_question.py", "--voice", VOICE,
                            "--text", s["question"], "--out", str(wav)], env=env, capture_output=True, text=True)
        print(r.stdout.strip() if r.returncode == 0 else r.stderr)

In [ ]:
!python {EXP}/t4/precompute_reference_embeddings.py --manifest {EXP}/configs/samples_hard_pilot.jsonl --cache-dir {CACHE}

In [ ]:
HARD = f"{EXP}/configs/samples_hard_pilot.jsonl"
ARGS = f"--reference-cache {CACHE} --output-root {RESULTS} --resume"
# Extremes first: 12 questions x {0 s, 8 s} = 24 runs (~1 min each). To add the other delays later,
# rerun with "--retrieval-delay 0 1 2 3 5 8": completed runs are skipped.
DELAYS = "0 8"
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {HARD} --retrieval-delay {DELAYS} --run-name pilot_hard {ARGS} --dry-run
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {HARD} --retrieval-delay {DELAYS} --run-name pilot_hard {ARGS}

In [ ]:
!python {EXP}/analysis/analyze_latency.py {RESULTS}/pilot_hard
!python {EXP}/analysis/export_annotation_sheet.py {RESULTS}/pilot_hard
import json
for r in map(json.loads, open(f"{RESULTS}/pilot_hard/results.jsonl")):
    print(f"--- {r['run_id']} | gold: {r['answer'][0]} | [RET] {r['retrieval_trigger_time']} s, injection {r['reference_injection_time']} s")
    print("   pre-RAG :", r["pre_rag_transcript"])
    print("   post-RAG:", r["post_rag_transcript"])

Annotation: download `pilot_hard/analysis/annotation_sheet.csv` from Drive, open it in Google Sheets and fill the label
columns (see `analysis/export_annotation_sheet.py`). More seeds later: rerun the run cell with
`--seeds 42424242 1 2` added; completed runs are skipped.